In [ ]:
import torch
import cv2
import numpy as np
from types import SimpleNamespace
from torchvision.transforms import Compose, Normalize, ToTensor
from lib.config import cfg, update_config
from models import pose_hrnet

import os
import cv2
import json
from glob import glob
import numpy as np

CONFIDENCE = 0.7

: 

Initialize HRNET 

In [ ]:
# Initialize all required config attributes
args = SimpleNamespace(
    cfg="experiments/coco/hrnet/w48_384x288_adam_lr1e-3.yaml",  # Path to YAML config
    opts=[],             # No command line overrides needed
    modelDir='',         # Not used for inference
    logDir='',           # Not used for inference
    dataDir='',          # Not used for inference
)

# Load and update configuration
update_config(cfg, args)

model = pose_hrnet.get_pose_net(cfg, is_train=False)

# Load pretrained weights
model.load_state_dict(
    torch.load(
        "models/pose_hrnet_w48_384x288.pth",
        map_location=torch.device('cpu')
    )
)
model.eval() 

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Resize and normalize Images

In [ ]:
def preprocess(image):
    # Resize to model's expected input size (width, height)
    image = cv2.resize(image, (288, 384))  # Note: OpenCV uses (width, height)
    
    # Normalize and convert to tensor
    transform = Compose([
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transform(image).unsqueeze(0).to(device)  # Add batch dimension

Input image to the model

In [ ]:
def run_inference(image_path):
    # Load and preprocess image
    image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    input_tensor = preprocess(image)
    
    # Run model
    with torch.no_grad():
        output = model(input_tensor)
    
    return output

Get and preprocess Keypoints for Json allocation

In [ ]:
def get_keypoints(heatmaps, original_size):
    """Convert heatmaps to specific body parts with shoulder midpoint as Zona 1 (when both shoulders detected)"""
    SELECTED_KEYPOINTS = [
        ("left_shoulder", "hombro_izq"),
        ("right_shoulder", "hombro_drc"),
        ("left_elbow", "codo_izq"),
        ("right_elbow", "codo_drc"),
        ("left_knee", "rodilla_izq"),
        ("right_knee", "rodilla_drc"),
        ("left_wrist", "mano_izq"),
        ("right_wrist", "mano_drc"),
        ("left_ankle", "pie_izq"),
        ("right_ankle", "pie_drc")
    ]
    
    heatmaps = heatmaps.squeeze(0).cpu().numpy()
    keypoints = []
    
    # Create a mapping from COCO names to index positions
    COCO_KEYPOINT_NAMES = [
        "nose", "left_eye", "right_eye", "left_ear", "right_ear",
        "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
        "left_wrist", "right_wrist", "left_hip", "right_hip",
        "left_knee", "right_knee", "left_ankle", "right_ankle"
    ]
    name_to_index = {name: i for i, name in enumerate(COCO_KEYPOINT_NAMES)}
    
    # First collect all points we need
    point_data = {}
    for coco_name, display_name in SELECTED_KEYPOINTS:
        i = name_to_index[coco_name]
        heatmap = heatmaps[i]
        max_val = heatmap.max()
        
        if max_val > CONFIDENCE:  # Only consider if some pixel was detected
            y, x = np.unravel_index(np.argmax(heatmap), heatmap.shape)
            x_scaled = x * (original_size[0] / heatmap.shape[1])
            y_scaled = y * (original_size[1] / heatmap.shape[0])
            
            point_data[coco_name] = {
                'x': x_scaled,
                'y': y_scaled,
                'class': display_name,
                'confidence': float(max_val)
            }
        else:
            point_data[coco_name] = {
                'x': None,
                'y': None,
                'class': "no_present",
                'confidence': None
            }
    print (point_data)
    # Handle shoulders - first check if we have any shoulder detections
    has_left_shoulder = 'left_shoulder' in point_data and point_data['left_shoulder']['class'] != "no_present"
    has_right_shoulder = 'right_shoulder' in point_data and point_data['right_shoulder']['class'] != "no_present"
    
    if has_left_shoulder and has_right_shoulder:
        # Both shoulders detected - calculate midpoint as Zona 1
        l_shoulder = point_data['left_shoulder']
        r_shoulder = point_data['right_shoulder']
        
        zona1_confidence = min(l_shoulder['confidence'], r_shoulder['confidence'])
        
        keypoints.append({
            'x': (l_shoulder['x'] + r_shoulder['x']) / 2,
            'y': (l_shoulder['y'] + r_shoulder['y']) / 2,
            'class': 'Zona1',
            'confidence': zona1_confidence
        })
        
    elif has_left_shoulder:
        # Only left shoulder detected - add it as Zona1
        keypoints.append({
            'x': point_data['left_shoulder']['x'],
            'y': point_data['left_shoulder']['y'],
            'class': 'Zona1',
            'confidence': point_data['left_shoulder']['confidence']
        })

    elif has_right_shoulder:
        # Only right shoulder detected - add it as Zona1
        keypoints.append({
            'x': point_data['right_shoulder']['x'],
            'y': point_data['right_shoulder']['y'],
            'class': 'Zona1',
            'confidence': point_data['right_shoulder']['confidence']
        })
        
    else:
        # No shoulders detected - add Zona1 as no_present
        keypoints.append({
            'x': None,
            'y': None,
            'class': "no_present",
            'confidence': None
        })
    
    # Add all other points (excluding shoulders which we've already handled)
    for coco_name, display_name in SELECTED_KEYPOINTS[2:]:  # Skip first two (shoulders)
        keypoints.append({
            'x': point_data[coco_name]['x'],
            'y': point_data[coco_name]['y'],
            'class': point_data[coco_name]['class'],
            'confidence': point_data[coco_name]['confidence']
        })
    
    return keypoints

Save data into the Json file

In [ ]:
def save_progress():
    """Save the current progress to JSON file"""
    with open(output_json_path, 'w') as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)
    print(f"Progress saved: {len(output_data)} images processed")

MAIN

In [ ]:
# Base paths
aligned_path = "/data/uabcvmsc/cvmsct01/CroppingMechanisms/newbornAligned"
validation_path = "/data/uabcvmsc/cvmsct01/DATASETS/NewbornDataset"

# Get all subfolders from 29 to 55
subfolders = sorted([f for f in os.listdir(aligned_path) 
                    if f.isdigit() and 29 <= int(f) <= 55])

# Output JSON path
output_json_path = "/data/uabcvmsc/cvmsct01/PostureDetection/HRNet-Human-Pose-Estimation/full_output/pose_annotations_07.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# Initialize or load existing JSON data
if os.path.exists(output_json_path):
    with open(output_json_path, 'r') as f:
        output_data = json.load(f)
else:
    output_data = []


for subfolder in subfolders:
    # Get all date subfolders within each numbered folder
    date_folders = [f for f in os.listdir(os.path.join(aligned_path, subfolder)) 
                   if os.path.isdir(os.path.join(aligned_path, subfolder, f))]
    
    for date_folder in date_folders:
        # Skip if this folder was already processed
        folder_processed = any(
            entry['aligned_image'].startswith(f"{subfolder}\\{date_folder}\\") 
            for entry in output_data
        )
        if folder_processed:
            print(f"Skipping already processed folder: {subfolder}/{date_folder}")
            continue
            
        print(f"\nProcessing {subfolder}/{date_folder}")
        folder_output = []
        
        # Get all aligned images in this date folder
        aligned_images = sorted(glob(os.path.join(aligned_path, subfolder, date_folder, "aligned_*VIS.jpeg")))
        
        # Inside the loop where we process each aligned image:
        for aligned_img in aligned_images:
            print(f"Processing {os.path.basename(aligned_img)}", end='... ')
            
            try:
                # Read aligned image
                original_image = cv2.imread(aligned_img)
                if original_image is None:
                    print("Failed to read image")
                    continue
                    
                original_size = (original_image.shape[1], original_image.shape[0])
                
                # Run inference
                output = run_inference(aligned_img)
                keypoints = get_keypoints(output, original_size)
                                
                # Get base filename and handle both formats:
                base_name = os.path.basename(aligned_img)
                
                # Remove 'aligned_' prefix and extension
                img_id = base_name[8:] 
                
                # Handle both filename formats:
                if '_VIS.jpeg' in img_id:
                    # Format: aligned_HM20240814213403_VIS.jpeg
                    img_id = img_id.replace('_VIS.jpeg', '')
                    thermal_rel = f"{subfolder}\\{date_folder}\\{img_id}.jpeg"
                else:
                    # Format: aligned_HM20240814213403.VIS.jpeg
                    img_id = img_id.replace('.VIS.jpeg', '')
                    thermal_rel = f"{subfolder}\\{date_folder}\\{img_id}.jpeg"
                
                aligned_rel = f"{subfolder}\\{date_folder}\\{base_name}"
                
                # Prepare annotations
                annotations = []
                for kp in keypoints:
                    annotation = {
                        "x": round(float(kp['x'])) if kp['x'] is not None else None,
                        "y": round(float(kp['y'])) if kp['y'] is not None else None,
                        "class": kp['class'],
                        "confidence": kp['confidence'],
                    }
                    annotations.append(annotation)
                                
                folder_output.append({
                    "thermal_image": thermal_rel,
                    "aligned_image": aligned_rel,
                    "anotaciones": annotations
                })
                
                print("Success")
                    
            except Exception as e:
                print(f"Error: {str(e)}")
                continue
        
        # Add this folder's results to main output and save
        output_data.extend(folder_output)
        save_progress()
        print(f"Finished processing {subfolder}/{date_folder} - {len(folder_output)} images")

print(f"\nFinal results: {len(output_data)} image sets processed")
print(f"Results saved to: {output_json_path}")